# Insights gráficos — Golds Squad 3

Este notebook centraliza análises visuais das tabelas Gold geradas no projeto.

Ele **não faz parte da pipeline produtiva**.

O objetivo é apoiar a validação das regras de negócio, levantar dúvidas para discussão com o gestor e facilitar a interpretação dos principais KPIs.

## Como usar este notebook

Cada seção representa uma tabela Gold.

Para cada Gold, o notebook apresenta:

- objetivo da análise;
- regra de negócio considerada;
- principais indicadores resumidos;
- tabela de apoio;
- gráficos principais;
- observações de interpretação.

## Importante

Este notebook é apenas analítico.

Ele não grava dados, não altera tabelas e não deve ser chamado pelos pipelines Bronze, Silver ou Gold.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Importa bibliotecas e define funções auxiliares para o notebook de insights.

from pyspark.sql.functions import (
    col,
    concat_ws,
    lpad,
    when
)

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "figure.figsize": (14, 6),
    "axes.grid": True,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9
})


def adicionar_ano_mes_label(df, ano_col, mes_col, label_col="ano_mes"):
    return df.withColumn(
        label_col,
        concat_ws("-", col(ano_col), lpad(col(mes_col).cast("string"), 2, "0"))
    )


def ler_gold(nome_gold):
    final_table = f"{TARGET_SCHEMA}.{nome_gold}"

    return read_sql_table(
        spark=spark,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD,
        table_name=final_table,
        sql_port=SQL_PORT
    )


def preparar_pdf(df, numeric_cols=None):
    pdf = df.toPandas()

    if numeric_cols:
        for coluna in numeric_cols:
            if coluna in pdf.columns:
                pdf[coluna] = pd.to_numeric(pdf[coluna], errors="coerce")

    return pdf


def fmt_int(valor):
    if pd.isna(valor):
        return "-"
    return f"{int(valor):,}".replace(",", ".")


def fmt_float(valor, casas=2):
    if pd.isna(valor):
        return "-"
    texto = f"{float(valor):,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_percent(valor, casas=2):
    return f"{fmt_float(valor, casas)}%"


def fmt_money(valor):
    return f"R$ {fmt_float(valor, 2)}"


def exibir_kpis(titulo, indicadores):
    cards_html = ""

    for nome, valor in indicadores.items():
        cards_html += f"""
        <div style="
            border: 1px solid #ddd;
            border-radius: 10px;
            padding: 14px;
            min-width: 190px;
            background-color: #fafafa;
        ">
            <div style="font-size: 13px; color: #666;">{nome}</div>
            <div style="font-size: 24px; font-weight: bold;">{valor}</div>
        </div>
        """

    displayHTML(f"""
    <h3>{titulo}</h3>
    <div style="
        display: flex;
        flex-wrap: wrap;
        gap: 12px;
        margin-bottom: 16px;
    ">
        {cards_html}
    </div>
    """)


def formatar_grafico(titulo, xlabel, ylabel, rotacao_x=45):
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotacao_x)
    plt.tight_layout()
    plt.show()


def grafico_barras(pdf, x_col, y_col, titulo, xlabel, ylabel, rotacao_x=45):
    plt.figure()
    plt.bar(pdf[x_col], pdf[y_col])
    formatar_grafico(titulo, xlabel, ylabel, rotacao_x)


def grafico_linha(pdf, x_col, y_col, titulo, xlabel, ylabel, label=None, rotacao_x=45):
    plt.figure()
    plt.plot(pdf[x_col], pdf[y_col], marker="o", label=label)

    if label:
        plt.legend()

    formatar_grafico(titulo, xlabel, ylabel, rotacao_x)


def grafico_linha_por_categoria(pdf, x_col, y_col, categoria_col, titulo, xlabel, ylabel):
    pivot = (
        pdf
        .pivot_table(
            index=x_col,
            columns=categoria_col,
            values=y_col,
            aggfunc="mean"
        )
        .sort_index()
    )

    plt.figure()

    for coluna in pivot.columns:
        plt.plot(
            pivot.index,
            pivot[coluna],
            marker="o",
            label=coluna
        )

    plt.legend()
    formatar_grafico(titulo, xlabel, ylabel)


def grafico_barras_empilhadas(pdf, x_col, col_base, col_topo, label_base, label_topo, titulo, xlabel, ylabel):
    plt.figure()

    plt.bar(
        pdf[x_col],
        pdf[col_base],
        label=label_base
    )

    plt.bar(
        pdf[x_col],
        pdf[col_topo],
        bottom=pdf[col_base],
        label=label_topo
    )

    plt.legend()
    formatar_grafico(titulo, xlabel, ylabel)


print("Funções auxiliares carregadas com sucesso.")

## Insight — Cadastros mensais de clientes

Tabela Gold:

`gold_ecommerce_clientes_cadastros_mensal`

Objetivo:

- visualizar a evolução mensal de novos clientes;
- acompanhar o crescimento acumulado da base;
- observar variações mês contra mês.

In [0]:
# Lê e prepara a Gold de cadastros mensais de clientes.

NOME_GOLD = "gold_ecommerce_clientes_cadastros_mensal"

df_cadastros_mensal = ler_gold(NOME_GOLD)

df_insight_cadastros_mensal = (
    adicionar_ano_mes_label(
        df_cadastros_mensal,
        ano_col="ano_cadastro",
        mes_col="mes_cadastro"
    )
    .select(
        "ano_mes",
        "data_referencia",
        "qtd_clientes_novos",
        "qtd_clientes_acumulado",
        "crescimento_mom_percentual"
    )
    .orderBy("data_referencia")
)

pdf_cadastros_mensal = preparar_pdf(
    df_insight_cadastros_mensal,
    numeric_cols=[
        "qtd_clientes_novos",
        "qtd_clientes_acumulado",
        "crescimento_mom_percentual"
    ]
)

exibir_kpis(
    "Resumo — Cadastros mensais",
    {
        "Meses analisados": fmt_int(len(pdf_cadastros_mensal)),
        "Total de clientes": fmt_int(pdf_cadastros_mensal["qtd_clientes_novos"].sum()),
        "Maior mês de cadastro": fmt_int(pdf_cadastros_mensal["qtd_clientes_novos"].max()),
        "Base acumulada final": fmt_int(pdf_cadastros_mensal["qtd_clientes_acumulado"].max())
    }
)

display(df_insight_cadastros_mensal)

In [0]:
# Gera gráficos de cadastros mensais.

grafico_linha(
    pdf_cadastros_mensal,
    x_col="ano_mes",
    y_col="qtd_clientes_novos",
    titulo="Evolução mensal de novos clientes",
    xlabel="Ano/Mês",
    ylabel="Quantidade de novos clientes"
)

grafico_linha(
    pdf_cadastros_mensal,
    x_col="ano_mes",
    y_col="qtd_clientes_acumulado",
    titulo="Evolução acumulada da base de clientes",
    xlabel="Ano/Mês",
    ylabel="Quantidade acumulada de clientes"
)

## Insight — Tempo médio mensal de entrega por transportadora

Tabela Gold:

`gold_ecommerce_rastreamento_entregas_tempo_medio_mensal`

Objetivo:

- visualizar o tempo médio mensal de entrega por transportadora;
- comparar o tempo médio real com o SLA prometido;
- identificar meses e transportadoras com maior desvio em relação ao SLA.

In [0]:
# Lê e prepara a Gold de tempo médio mensal de entrega.

NOME_GOLD = "gold_ecommerce_rastreamento_entregas_tempo_medio_mensal"

df_tempo_medio_mensal = ler_gold(NOME_GOLD)

df_insight_tempo_medio_mensal = (
    adicionar_ano_mes_label(
        df_tempo_medio_mensal,
        ano_col="ano_entrega",
        mes_col="mes_entrega"
    )
    .select(
        "ano_mes",
        "ano_entrega",
        "mes_entrega",
        "id_transportadora",
        "qtd_pedidos_entregues",
        "tempo_medio_entrega_dias",
        "sla_prometido_dias",
        "desvio_medio_sla_dias"
    )
    .orderBy("ano_entrega", "mes_entrega", "id_transportadora")
)

pdf_tempo_medio = preparar_pdf(
    df_insight_tempo_medio_mensal,
    numeric_cols=[
        "qtd_pedidos_entregues",
        "tempo_medio_entrega_dias",
        "sla_prometido_dias",
        "desvio_medio_sla_dias"
    ]
)

pdf_tempo_medio["transportadora"] = "Transportadora " + pdf_tempo_medio["id_transportadora"].astype(str)

exibir_kpis(
    "Resumo — Tempo médio de entrega",
    {
        "Pedidos entregues": fmt_int(pdf_tempo_medio["qtd_pedidos_entregues"].sum()),
        "Tempo médio geral": f'{fmt_float(pdf_tempo_medio["tempo_medio_entrega_dias"].mean())} dias',
        "SLA prometido": f'{fmt_float(pdf_tempo_medio["sla_prometido_dias"].mean())} dias',
        "Desvio médio geral": f'{fmt_float(pdf_tempo_medio["desvio_medio_sla_dias"].mean())} dias'
    }
)

display(df_insight_tempo_medio_mensal)

In [0]:
# Gera gráficos de tempo médio e desvio em relação ao SLA.

grafico_linha_por_categoria(
    pdf_tempo_medio,
    x_col="ano_mes",
    y_col="tempo_medio_entrega_dias",
    categoria_col="transportadora",
    titulo="Tempo médio mensal de entrega por transportadora",
    xlabel="Ano/Mês",
    ylabel="Tempo médio de entrega em dias"
)

pivot_desvio_sla = (
    pdf_tempo_medio
    .pivot_table(
        index="ano_mes",
        columns="transportadora",
        values="desvio_medio_sla_dias",
        aggfunc="mean"
    )
    .sort_index()
)

plt.figure()

for coluna in pivot_desvio_sla.columns:
    plt.plot(
        pivot_desvio_sla.index,
        pivot_desvio_sla[coluna],
        marker="o",
        label=coluna
    )

plt.axhline(0, linestyle="--")
plt.legend()
formatar_grafico(
    titulo="Desvio médio em relação ao SLA por transportadora",
    xlabel="Ano/Mês",
    ylabel="Desvio médio em dias"
)

## Insight — Distribuição de clientes por estado

Tabela Gold:

`gold_ecommerce_clientes_distribuicao_estado`

Objetivo:

- visualizar a quantidade de clientes por estado;
- comparar a participação percentual de cada estado na base;
- identificar concentração geográfica da base.

In [0]:
# Lê e prepara a Gold de distribuição de clientes por estado.

NOME_GOLD = "gold_ecommerce_clientes_distribuicao_estado"

df_distribuicao_estado = ler_gold(NOME_GOLD)

df_insight_distribuicao_estado = (
    df_distribuicao_estado
    .select(
        "estado",
        "qtd_clientes",
        "percentual_clientes"
    )
    .orderBy(col("qtd_clientes").desc())
)

pdf_distribuicao_estado = preparar_pdf(
    df_insight_distribuicao_estado,
    numeric_cols=[
        "qtd_clientes",
        "percentual_clientes"
    ]
)

exibir_kpis(
    "Resumo — Distribuição por estado",
    {
        "Estados encontrados": fmt_int(len(pdf_distribuicao_estado)),
        "Total de clientes": fmt_int(pdf_distribuicao_estado["qtd_clientes"].sum()),
        "Maior concentração": pdf_distribuicao_estado.iloc[0]["estado"],
        "% maior estado": fmt_percent(pdf_distribuicao_estado.iloc[0]["percentual_clientes"])
    }
)

display(df_insight_distribuicao_estado)

In [0]:
# Gera gráficos de distribuição de clientes por estado.

grafico_barras(
    pdf_distribuicao_estado,
    x_col="estado",
    y_col="qtd_clientes",
    titulo="Quantidade de clientes por estado",
    xlabel="Estado",
    ylabel="Quantidade de clientes",
    rotacao_x=0
)

grafico_barras(
    pdf_distribuicao_estado,
    x_col="estado",
    y_col="percentual_clientes",
    titulo="Percentual de clientes por estado",
    xlabel="Estado",
    ylabel="Percentual de clientes (%)",
    rotacao_x=0
)

## Insight — Clientes ativos por cohort

Tabela Gold:

`gold_ecommerce_clientes_ativos_cohort`

Objetivo:

- visualizar o percentual de clientes ativos por mês de cadastro;
- comparar a quantidade de clientes ativos e inativos por cohort;
- identificar cohorts com maior ou menor atividade recente.

Observação:

Cliente ativo, nesta Gold, significa cliente com pelo menos 1 pedido nos últimos 60 dias, usando como referência a maior data de pedido disponível na base.

In [0]:
# Lê e prepara a Gold de clientes ativos por cohort.

NOME_GOLD = "gold_ecommerce_clientes_ativos_cohort"

df_clientes_ativos_cohort = ler_gold(NOME_GOLD)

df_insight_clientes_ativos_cohort = (
    adicionar_ano_mes_label(
        df_clientes_ativos_cohort,
        ano_col="ano_cadastro",
        mes_col="mes_cadastro"
    )
    .select(
        "ano_mes",
        "data_cohort",
        "qtd_clientes_cadastrados",
        "qtd_clientes_ativos",
        "qtd_clientes_inativos",
        "percentual_clientes_ativos"
    )
    .orderBy("data_cohort")
)

pdf_clientes_ativos = preparar_pdf(
    df_insight_clientes_ativos_cohort,
    numeric_cols=[
        "qtd_clientes_cadastrados",
        "qtd_clientes_ativos",
        "qtd_clientes_inativos",
        "percentual_clientes_ativos"
    ]
)

exibir_kpis(
    "Resumo — Clientes ativos por cohort",
    {
        "Clientes cadastrados": fmt_int(pdf_clientes_ativos["qtd_clientes_cadastrados"].sum()),
        "Clientes ativos": fmt_int(pdf_clientes_ativos["qtd_clientes_ativos"].sum()),
        "Clientes inativos": fmt_int(pdf_clientes_ativos["qtd_clientes_inativos"].sum()),
        "% ativo médio": fmt_percent(pdf_clientes_ativos["percentual_clientes_ativos"].mean())
    }
)

display(df_insight_clientes_ativos_cohort)

In [0]:
# Gera gráficos de clientes ativos por cohort.

grafico_linha(
    pdf_clientes_ativos,
    x_col="ano_mes",
    y_col="percentual_clientes_ativos",
    titulo="Percentual de clientes ativos por cohort",
    xlabel="Cohort de cadastro",
    ylabel="Clientes ativos (%)"
)

grafico_barras_empilhadas(
    pdf_clientes_ativos,
    x_col="ano_mes",
    col_base="qtd_clientes_ativos",
    col_topo="qtd_clientes_inativos",
    label_base="Clientes ativos",
    label_topo="Clientes inativos",
    titulo="Clientes ativos e inativos por cohort",
    xlabel="Cohort de cadastro",
    ylabel="Quantidade de clientes"
)

## Insight — LTV por cohort de clientes

Tabela Gold:

`gold_ecommerce_clientes_ltv_cohort`

Objetivo:

- visualizar o LTV médio dos clientes por mês de cadastro;
- acompanhar a receita total gerada por cada cohort;
- comparar cohorts com maior ou menor valor médio gerado.

Observação:

Esta Gold considera apenas pedidos com status `ENTREGUE`, pois representam receita realizada confirmada.

In [0]:
# Lê e prepara a Gold de LTV por cohort de clientes.

NOME_GOLD = "gold_ecommerce_clientes_ltv_cohort"

df_ltv_cohort = ler_gold(NOME_GOLD)

df_insight_ltv_cohort = (
    adicionar_ano_mes_label(
        df_ltv_cohort,
        ano_col="ano_cadastro",
        mes_col="mes_cadastro"
    )
    .select(
        "ano_mes",
        "data_cohort",
        "qtd_clientes_cadastrados",
        "qtd_clientes_com_pedido_entregue",
        "qtd_clientes_sem_pedido_entregue",
        "qtd_pedidos_entregues",
        "receita_total_cohort",
        "ltv_medio_clientes_cadastrados",
        "ltv_medio_clientes_com_pedido_entregue"
    )
    .orderBy("data_cohort")
)

pdf_ltv_cohort = preparar_pdf(
    df_insight_ltv_cohort,
    numeric_cols=[
        "qtd_clientes_cadastrados",
        "qtd_clientes_com_pedido_entregue",
        "qtd_clientes_sem_pedido_entregue",
        "qtd_pedidos_entregues",
        "receita_total_cohort",
        "ltv_medio_clientes_cadastrados",
        "ltv_medio_clientes_com_pedido_entregue"
    ]
)

exibir_kpis(
    "Resumo — LTV por cohort",
    {
        "Receita total": fmt_money(pdf_ltv_cohort["receita_total_cohort"].sum()),
        "Pedidos entregues": fmt_int(pdf_ltv_cohort["qtd_pedidos_entregues"].sum()),
        "LTV médio cadastrado": fmt_money(pdf_ltv_cohort["ltv_medio_clientes_cadastrados"].mean()),
        "LTV médio comprador": fmt_money(pdf_ltv_cohort["ltv_medio_clientes_com_pedido_entregue"].mean())
    }
)

display(df_insight_ltv_cohort)

In [0]:
# Gera gráficos de LTV e receita por cohort.

plt.figure()

plt.plot(
    pdf_ltv_cohort["ano_mes"],
    pdf_ltv_cohort["ltv_medio_clientes_cadastrados"],
    marker="o",
    label="LTV médio por cliente cadastrado"
)

plt.plot(
    pdf_ltv_cohort["ano_mes"],
    pdf_ltv_cohort["ltv_medio_clientes_com_pedido_entregue"],
    marker="o",
    label="LTV médio por cliente com pedido entregue"
)

plt.legend()
formatar_grafico(
    titulo="LTV médio por cohort de clientes",
    xlabel="Cohort de cadastro",
    ylabel="LTV médio"
)

grafico_barras(
    pdf_ltv_cohort,
    x_col="ano_mes",
    y_col="receita_total_cohort",
    titulo="Receita total por cohort de clientes",
    xlabel="Cohort de cadastro",
    ylabel="Receita total"
)

## Insight — Perfil socioeconômico de clientes por UF

Tabela Gold:

`gold_ecommerce_clientes_perfil_socioeconomico_uf`

Objetivo:

- visualizar a distribuição de clientes por UF;
- comparar a base de clientes com a renda média per capita da UF;
- validar o percentual de match com a base IBGE.

Observação:

Esta Gold usa o estado do endereço principal do cliente e cruza com a Silver `ibge_renda_uf`.

In [0]:
# Lê e prepara a Gold de perfil socioeconômico por UF.

NOME_GOLD = "gold_ecommerce_clientes_perfil_socioeconomico_uf"

df_perfil_socioeconomico_uf = ler_gold(NOME_GOLD)

df_insight_perfil_socioeconomico_uf = (
    df_perfil_socioeconomico_uf
    .select(
        "estado",
        "nome_uf",
        "qtd_clientes",
        "percentual_clientes",
        "qtd_clientes_com_match_ibge",
        "qtd_clientes_sem_match_ibge",
        "percentual_match_ibge",
        "renda_media_per_capita",
        "ano_referencia",
        "fonte"
    )
    .orderBy(col("qtd_clientes").desc())
)

pdf_perfil_uf = preparar_pdf(
    df_insight_perfil_socioeconomico_uf,
    numeric_cols=[
        "qtd_clientes",
        "percentual_clientes",
        "qtd_clientes_com_match_ibge",
        "qtd_clientes_sem_match_ibge",
        "percentual_match_ibge",
        "renda_media_per_capita"
    ]
)

exibir_kpis(
    "Resumo — Perfil socioeconômico por UF",
    {
        "UFs analisadas": fmt_int(len(pdf_perfil_uf)),
        "Total de clientes": fmt_int(pdf_perfil_uf["qtd_clientes"].sum()),
        "Match IBGE médio": fmt_percent(pdf_perfil_uf["percentual_match_ibge"].mean()),
        "Renda média simples": fmt_money(pdf_perfil_uf["renda_media_per_capita"].mean())
    }
)

display(df_insight_perfil_socioeconomico_uf)

In [0]:
# Gera gráficos de perfil socioeconômico por UF.

grafico_barras(
    pdf_perfil_uf,
    x_col="estado",
    y_col="qtd_clientes",
    titulo="Quantidade de clientes por UF",
    xlabel="UF",
    ylabel="Quantidade de clientes",
    rotacao_x=0
)

grafico_barras(
    pdf_perfil_uf,
    x_col="estado",
    y_col="renda_media_per_capita",
    titulo="Renda média per capita por UF",
    xlabel="UF",
    ylabel="Renda média per capita",
    rotacao_x=0
)

grafico_barras(
    pdf_perfil_uf,
    x_col="estado",
    y_col="percentual_match_ibge",
    titulo="Percentual de match com a base IBGE por UF",
    xlabel="UF",
    ylabel="Match IBGE (%)",
    rotacao_x=0
)

## Insight — Eventos de entrega em feriados

Tabela Gold:

`gold_ecommerce_rastreamento_entregas_feriados_sucesso`

Objetivo:

- comparar o percentual de eventos com status `entregue` em feriados e fora de feriados;
- comparar o volume de eventos registrados em feriados e fora de feriados;
- observar se feriados concentram menos eventos finalizados como entrega.

Observação importante:

Esta análise é baseada em **eventos de rastreamento**, não em pedidos únicos.

Portanto, o campo originalmente chamado de `taxa_sucesso_percentual` deve ser interpretado como **percentual de eventos com status `entregue`**, e não como taxa final de sucesso da operação.

In [0]:
# Lê e prepara a Gold de eventos de entrega em feriados.

NOME_GOLD = "gold_ecommerce_rastreamento_entregas_feriados_sucesso"

df_feriados_sucesso = ler_gold(NOME_GOLD)

df_insight_feriados_sucesso = (
    adicionar_ano_mes_label(
        df_feriados_sucesso,
        ano_col="ano_evento",
        mes_col="mes_evento"
    )
    .withColumn(
        "tipo_dia",
        when(col("evento_em_feriado") == 1, "Feriado").otherwise("Não feriado")
    )
    .select(
        "ano_mes",
        "ano_evento",
        "mes_evento",
        "tipo_dia",
        "evento_em_feriado",
        "qtd_eventos",
        "qtd_eventos_entregues",
        col("qtd_eventos_nao_entregues").alias("qtd_eventos_outros_status"),
        col("taxa_sucesso_percentual").alias("percentual_eventos_entregues")
    )
    .orderBy("ano_evento", "mes_evento", "evento_em_feriado")
)

pdf_feriados_sucesso = preparar_pdf(
    df_insight_feriados_sucesso,
    numeric_cols=[
        "qtd_eventos",
        "qtd_eventos_entregues",
        "qtd_eventos_outros_status",
        "percentual_eventos_entregues"
    ]
)

exibir_kpis(
    "Resumo — Eventos em feriados",
    {
        "Eventos analisados": fmt_int(pdf_feriados_sucesso["qtd_eventos"].sum()),
        "Eventos entregues": fmt_int(pdf_feriados_sucesso["qtd_eventos_entregues"].sum()),
        "Eventos outros status": fmt_int(pdf_feriados_sucesso["qtd_eventos_outros_status"].sum()),
        "% eventos entregues médio": fmt_percent(pdf_feriados_sucesso["percentual_eventos_entregues"].mean())
    }
)

display(df_insight_feriados_sucesso)

In [0]:
# Gera gráficos de eventos em feriados e fora de feriados.

grafico_linha_por_categoria(
    pdf_feriados_sucesso,
    x_col="ano_mes",
    y_col="percentual_eventos_entregues",
    categoria_col="tipo_dia",
    titulo="Percentual de eventos com status entregue em feriados e fora de feriados",
    xlabel="Ano/Mês",
    ylabel="Eventos com status entregue (%)"
)

grafico_linha_por_categoria(
    pdf_feriados_sucesso,
    x_col="ano_mes",
    y_col="qtd_eventos",
    categoria_col="tipo_dia",
    titulo="Volume de eventos em feriados e fora de feriados",
    xlabel="Ano/Mês",
    ylabel="Quantidade de eventos"
)

## Insight — Problemas mensais em entregas

Tabela Gold:

`gold_ecommerce_rastreamento_entregas_problemas_mensal`

Objetivo:

- visualizar a quantidade de pedidos com problema logístico por mês;
- comparar o percentual de pedidos com problema por transportadora;
- identificar meses e transportadoras com maior concentração de problemas.

Observação:

Nesta Gold, um pedido é considerado com problema quando algum evento de rastreamento possui observação contendo `problemas na malha`.

A análise é consolidada por pedido, mês e transportadora.

In [0]:
# Lê e prepara a Gold de problemas mensais em entregas.

NOME_GOLD = "gold_ecommerce_rastreamento_entregas_problemas_mensal"

df_problemas_mensal = ler_gold(NOME_GOLD)

df_insight_problemas_mensal = (
    adicionar_ano_mes_label(
        df_problemas_mensal,
        ano_col="ano_evento",
        mes_col="mes_evento"
    )
    .select(
        "ano_mes",
        "ano_evento",
        "mes_evento",
        "id_transportadora",
        "qtd_pedidos",
        "qtd_pedidos_com_problema",
        "qtd_pedidos_sem_problema",
        "percentual_pedidos_com_problema"
    )
    .orderBy("ano_evento", "mes_evento", "id_transportadora")
)

pdf_problemas_mensal = preparar_pdf(
    df_insight_problemas_mensal,
    numeric_cols=[
        "qtd_pedidos",
        "qtd_pedidos_com_problema",
        "qtd_pedidos_sem_problema",
        "percentual_pedidos_com_problema"
    ]
)

pdf_problemas_mensal["transportadora"] = "Transportadora " + pdf_problemas_mensal["id_transportadora"].astype(str)

exibir_kpis(
    "Resumo — Problemas mensais",
    {
        "Pedidos analisados": fmt_int(pdf_problemas_mensal["qtd_pedidos"].sum()),
        "Pedidos com problema": fmt_int(pdf_problemas_mensal["qtd_pedidos_com_problema"].sum()),
        "Pedidos sem problema": fmt_int(pdf_problemas_mensal["qtd_pedidos_sem_problema"].sum()),
        "% problema médio": fmt_percent(pdf_problemas_mensal["percentual_pedidos_com_problema"].mean())
    }
)

display(df_insight_problemas_mensal)

In [0]:
# Gera gráficos de problemas mensais em entregas.

pivot_problemas = (
    pdf_problemas_mensal
    .pivot_table(
        index="ano_mes",
        columns="transportadora",
        values="qtd_pedidos_com_problema",
        aggfunc="sum"
    )
    .fillna(0)
    .sort_index()
)

pivot_problemas.plot(kind="bar", figsize=(14, 6))
formatar_grafico(
    titulo="Pedidos com problema logístico por mês e transportadora",
    xlabel="Ano/Mês",
    ylabel="Quantidade de pedidos com problema"
)

grafico_linha_por_categoria(
    pdf_problemas_mensal,
    x_col="ano_mes",
    y_col="percentual_pedidos_com_problema",
    categoria_col="transportadora",
    titulo="Percentual de pedidos com problema logístico por transportadora",
    xlabel="Ano/Mês",
    ylabel="Pedidos com problema (%)"
)

## Insight — SLA mensal de entregas

Tabela Gold:

`gold_ecommerce_rastreamento_entregas_sla_mensal`

Objetivo:

- comparar pedidos entregues dentro e fora do SLA;
- acompanhar o percentual de entregas dentro do SLA por transportadora;
- identificar meses e transportadoras com maior volume de entregas fora do prazo.

Observação:

Nesta Gold, o SLA prometido considerado é de 7 dias.

Pedidos fora do SLA representam entregas cujo tempo em trânsito foi maior que o SLA prometido.

In [0]:
# Lê e prepara a Gold de SLA mensal de entregas.

NOME_GOLD = "gold_ecommerce_rastreamento_entregas_sla_mensal"

df_sla_mensal = ler_gold(NOME_GOLD)

df_insight_sla_mensal = (
    adicionar_ano_mes_label(
        df_sla_mensal,
        ano_col="ano_entrega",
        mes_col="mes_entrega"
    )
    .select(
        "ano_mes",
        "ano_entrega",
        "mes_entrega",
        "id_transportadora",
        "qtd_pedidos_entregues",
        "qtd_pedidos_dentro_sla",
        "qtd_pedidos_fora_sla",
        "percentual_dentro_sla",
        "percentual_fora_sla",
        "sla_prometido_dias"
    )
    .orderBy("ano_entrega", "mes_entrega", "id_transportadora")
)

pdf_sla_mensal = preparar_pdf(
    df_insight_sla_mensal,
    numeric_cols=[
        "qtd_pedidos_entregues",
        "qtd_pedidos_dentro_sla",
        "qtd_pedidos_fora_sla",
        "percentual_dentro_sla",
        "percentual_fora_sla",
        "sla_prometido_dias"
    ]
)

pdf_sla_mensal["transportadora"] = "Transportadora " + pdf_sla_mensal["id_transportadora"].astype(str)

exibir_kpis(
    "Resumo — SLA mensal",
    {
        "Pedidos entregues": fmt_int(pdf_sla_mensal["qtd_pedidos_entregues"].sum()),
        "Dentro do SLA": fmt_int(pdf_sla_mensal["qtd_pedidos_dentro_sla"].sum()),
        "Fora do SLA": fmt_int(pdf_sla_mensal["qtd_pedidos_fora_sla"].sum()),
        "% dentro SLA médio": fmt_percent(pdf_sla_mensal["percentual_dentro_sla"].mean())
    }
)

display(df_insight_sla_mensal)

In [0]:
# Gera gráficos de SLA mensal.

pdf_sla_total_mensal = (
    pdf_sla_mensal
    .groupby("ano_mes", as_index=False)
    .agg({
        "qtd_pedidos_dentro_sla": "sum",
        "qtd_pedidos_fora_sla": "sum"
    })
    .sort_values("ano_mes")
)

grafico_barras_empilhadas(
    pdf_sla_total_mensal,
    x_col="ano_mes",
    col_base="qtd_pedidos_dentro_sla",
    col_topo="qtd_pedidos_fora_sla",
    label_base="Dentro do SLA",
    label_topo="Fora do SLA",
    titulo="Pedidos dentro e fora do SLA por mês",
    xlabel="Ano/Mês",
    ylabel="Quantidade de pedidos entregues"
)

grafico_linha_por_categoria(
    pdf_sla_mensal,
    x_col="ano_mes",
    y_col="percentual_dentro_sla",
    categoria_col="transportadora",
    titulo="Percentual de entregas dentro do SLA por transportadora",
    xlabel="Ano/Mês",
    ylabel="Entregas dentro do SLA (%)"
)

pivot_fora_sla = (
    pdf_sla_mensal
    .pivot_table(
        index="ano_mes",
        columns="transportadora",
        values="qtd_pedidos_fora_sla",
        aggfunc="sum"
    )
    .fillna(0)
    .sort_index()
)

pivot_fora_sla.plot(kind="bar", figsize=(14, 6))
formatar_grafico(
    titulo="Pedidos fora do SLA por mês e transportadora",
    xlabel="Ano/Mês",
    ylabel="Quantidade de pedidos fora do SLA"
)

## Insight — Volume mensal de entregas por estado com MoM

Tabela Gold:

`gold_ecommerce_rastreamento_entregas_volume_estado_mom`

Objetivo:

- visualizar o volume mensal de pedidos entregues por estado;
- acompanhar a variação mês contra mês do volume de entregas;
- identificar estados com crescimento ou queda relevante nas entregas.

Observação:

Esta Gold considera apenas pedidos com entrega concluída.

A métrica MoM compara o volume de entregas de um mês com o volume do mês anterior dentro do mesmo estado.

In [0]:
# Lê e prepara a Gold de volume mensal de entregas por estado com MoM.

NOME_GOLD = "gold_ecommerce_rastreamento_entregas_volume_estado_mom"

df_volume_estado_mom = ler_gold(NOME_GOLD)

df_insight_volume_estado_mom = (
    adicionar_ano_mes_label(
        df_volume_estado_mom,
        ano_col="ano_entrega",
        mes_col="mes_entrega"
    )
    .select(
        "ano_mes",
        "estado",
        "ano_entrega",
        "mes_entrega",
        "qtd_pedidos_entregues",
        "qtd_pedidos_entregues_mes_anterior",
        "variacao_absoluta_mom",
        "crescimento_mom_percentual"
    )
    .orderBy("estado", "ano_entrega", "mes_entrega")
)

pdf_volume_estado_mom = preparar_pdf(
    df_insight_volume_estado_mom,
    numeric_cols=[
        "qtd_pedidos_entregues",
        "qtd_pedidos_entregues_mes_anterior",
        "variacao_absoluta_mom",
        "crescimento_mom_percentual"
    ]
)

maior_variacao = pdf_volume_estado_mom.loc[
    pdf_volume_estado_mom["variacao_absoluta_mom"].abs().idxmax()
]

exibir_kpis(
    "Resumo — Volume por estado com MoM",
    {
        "Pedidos entregues": fmt_int(pdf_volume_estado_mom["qtd_pedidos_entregues"].sum()),
        "Estados analisados": fmt_int(pdf_volume_estado_mom["estado"].nunique()),
        "Maior variação abs.": fmt_int(maior_variacao["variacao_absoluta_mom"]),
        "Estado/mês destaque": f'{maior_variacao["estado"]} - {maior_variacao["ano_mes"]}'
    }
)

display(df_insight_volume_estado_mom)

In [0]:
# Gera gráficos de volume mensal por estado e variação MoM.

grafico_linha_por_categoria(
    pdf_volume_estado_mom,
    x_col="ano_mes",
    y_col="qtd_pedidos_entregues",
    categoria_col="estado",
    titulo="Volume mensal de pedidos entregues por estado",
    xlabel="Ano/Mês",
    ylabel="Quantidade de pedidos entregues"
)

grafico_linha_por_categoria(
    pdf_volume_estado_mom,
    x_col="ano_mes",
    y_col="crescimento_mom_percentual",
    categoria_col="estado",
    titulo="Crescimento MoM percentual de entregas por estado",
    xlabel="Ano/Mês",
    ylabel="Crescimento MoM (%)"
)

pdf_variacoes_relevantes = (
    pdf_volume_estado_mom
    .assign(abs_variacao=pdf_volume_estado_mom["variacao_absoluta_mom"].abs())
    .sort_values("abs_variacao", ascending=False)
    .head(10)
    .copy()
)

pdf_variacoes_relevantes["estado_mes"] = (
    pdf_variacoes_relevantes["estado"] + " - " + pdf_variacoes_relevantes["ano_mes"]
)

grafico_barras(
    pdf_variacoes_relevantes,
    x_col="estado_mes",
    y_col="variacao_absoluta_mom",
    titulo="Maiores variações absolutas no volume de entregas",
    xlabel="Estado/Mês",
    ylabel="Variação absoluta MoM"
)

# Fechamento

Este notebook consolida os principais gráficos das Golds da Squad 3.

Ele deve ser usado para:

- revisar visualmente os resultados;
- apoiar discussões com o gestor;
- identificar dúvidas de regra de negócio;
- selecionar quais gráficos podem virar dashboard no Looker.

Ele não deve ser usado para gravação, carga ou atualização de dados.